# SIM V3 · Indoor · Phase B + C + D — one run
Dataset → train → validate → export, top-to-bottom in a **single Colab session on local `/content`** (no per-step Drive reads, so nothing gets lost to async sync). Results are saved to Drive **once**, verified, at the end. Pick a **GPU runtime**.

### 1 · Clone the repo + mount Drive

In [ ]:
# /content is wiped on every runtime restart, so (re)clone the code here.
import os, getpass
REPO_ROOT = '/content/indoor-walk-test'
if not os.path.isdir(os.path.join(REPO_ROOT, 'Physics Engine')):
    tok = getpass.getpass('GitHub token (repo read): ')
    os.system(f'git clone --depth 1 https://{tok}@github.com/cgm2179/indoor-walk-test.git "{REPO_ROOT}"')
try:
    from google.colab import drive; drive.mount('/content/drive')   # only for the final save
except ModuleNotFoundError:
    pass
print('repo present:', os.path.isdir(os.path.join(REPO_ROOT, 'Physics Engine', '2D', 'SIM V3')))

### 2 · Put SIM V3 on the path

In [ ]:
import sys, os
SIMV3 = os.path.join(REPO_ROOT, 'Physics Engine', '2D', 'SIM V3')
assert os.path.exists(os.path.join(SIMV3, '_bootstrap.py')), f'clone missing — re-run cell 1: {SIMV3}'
sys.path.insert(0, SIMV3); os.chdir(SIMV3)
import torch; print('SIM V3 =', SIMV3, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

### 3 · Parameters (everything local — reliable)

In [ ]:
BANDS = ['LTE_B71_617', 'LTE_B13_751', 'LTE_B2_1960', 'NR_n41_2506', 'NR_n77_3700', 'WiFi_2G4', 'WiFi_5G']
SCENE, N_TX, BOXES_PER_FIELD, BOX, N_PER_WAVELENGTH, REGION_M = 'indoor', 6, 80, 128, 8, 40
OUT  = '/content/fw_data_indoor'          # LOCAL disk: fast, reliable, no Drive-sync loss
CKPT = '/content/fw_unet2d_indoor.pt'
EPOCHS, BASE, BS = 80, 32, 16
# (trim BANDS to 1-2 for a quick smoke test; full 7-band gen is ~15 min on GPU)

### 4 · GPU acceleration (CuPy) — FDTD time loop on the GPU

In [ ]:
# --- GPU FDTD (CuPy): reuse FullWaveScene setup, run only the time loop on GPU ---
import numpy as np, math
import fw_dataset
from fullwave2d import FullWaveScene

C0 = 299_792_458.0
_cpu_run_field = fw_dataset._run_field          # keep original (fallback + parity check)

USE_GPU = False
try:
    import cupy as cp
    USE_GPU = cp.cuda.runtime.getDeviceCount() > 0
except Exception:
    try:  # Colab GPU runtime usually ships CuPy; install the CUDA-12 wheel if not
        import subprocess, sys
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'cupy-cuda12x'], check=True)
        import cupy as cp
        USE_GPU = cp.cuda.runtime.getDeviceCount() > 0
    except Exception as e:
        print('CuPy unavailable -> staying on CPU:', e)

def _laplacian_gpu(u, inv_h2):                  # matches Spatial_Physics.laplacian (np.roll)
    lap = cp.zeros_like(u)
    for ax in range(u.ndim):
        lap += cp.roll(u, 1, axis=ax) + cp.roll(u, -1, axis=ax)
    lap -= 2.0 * u.ndim * u
    return lap * inv_h2

def _run_field_gpu(classes, h, tx_ij, f_mhz, crossings):
    sim = FullWaveScene(classes, h, f_mhz, tx_ij, source='cw')     # identical CPU setup
    steps = int(round(crossings * max(classes.shape) * h / C0 / sim.dt))
    dt, f0 = sim.dt, sim.f0
    omega = 2.0 * np.pi * f0
    inv_h2 = float(sim.inv_h2)
    u      = cp.asarray(sim.u)                  # move only the fields the loop touches
    u_prev = cp.asarray(sim.u_prev)
    cdt2   = cp.asarray(sim.cdt2)
    inv1pa = cp.asarray(sim._inv1pa)
    _1ma   = cp.asarray(sim._1ma)
    damp   = cp.asarray(sim.damp)
    rigid  = cp.asarray(sim.rigid)
    src    = sim.src_idx
    warmup = int(0.6 * steps)                   # same as _run_field's simulate() call
    period_steps = max(1, int(round((1.0 / f0) / dt)))
    win_start = max(warmup, steps - 2 * period_steps)             # phasor_periods = 2
    acc = cp.zeros(u.shape, cp.complex128); n_win = 0
    for k in range(steps):                      # mirrors FullWaveScene.step() exactly
        lap = _laplacian_gpu(u, inv_h2)
        u_next = (2.0 * u - _1ma * u_prev + cdt2 * lap) * inv1pa
        u_next[src] += math.sin(omega * (k * dt))                # cw() soft source, t=k*dt
        u_next[rigid] = 0.0                     # perfect reflectors
        u_next *= damp                          # absorbing sponge
        u_prev = u * damp
        u = u_next
        if k >= win_start:                      # on-the-fly single-freq DFT (u at (k+1)dt)
            acc += u * complex(np.exp(-1j * omega * (k + 1) * dt))
            n_win += 1
    if not bool(cp.isfinite(u).all()):
        raise FloatingPointError('field blew up (GPU)')
    return cp.asnumpy((2.0 / max(n_win, 1)) * acc)               # complex phasor U, on CPU

if USE_GPU:
    fw_dataset._run_field = _run_field_gpu       # generate() -> _indoor_field -> this
    name = cp.cuda.runtime.getDeviceProperties(0)['name'].decode()
    # parity vs CPU on a tiny field so you can trust the port
    rng = np.random.default_rng(0)
    test = ((rng.random((96, 96)) < 0.12).astype(np.int8) * 2)   # sparse concrete
    Ucpu = _cpu_run_field(test, 0.06, (48, 48), 617.0, 1.2)
    Ugpu = _run_field_gpu(test, 0.06, (48, 48), 617.0, 1.2)
    rel = float(np.abs(Ugpu - Ucpu).max() / (np.abs(Ucpu).max() + 1e-30))
    print(f'GPU FDTD ON -> {name} | parity max|dU|/|U| = {rel:.2e} (want < 1e-6)')
else:
    print('GPU FDTD OFF -> CPU _run_field (pick a GPU runtime for the speed-up).')


### 5 · Phase B — generate the dataset (FDTD)

In [ ]:
import fw_dataset
fw_dataset.generate(BANDS, scene=SCENE, n_tx=N_TX, boxes_per_field=BOXES_PER_FIELD, box=BOX,
                    n_per_wavelength=N_PER_WAVELENGTH, region_m=REGION_M, out_dir=OUT, seed=1)

### 6 · Phase C — train the U-Net surrogate (AMP)

In [ ]:
import glob, fw_unet2d
print(len(glob.glob(OUT + '/shard_*.npz')), 'shards at', OUT)
model, best = fw_unet2d.train(OUT, epochs=EPOCHS, base=BASE, bs=BS, out=CKPT)
print('best val_mse =', best)

### 7 · Phase D — g2 validation + ONNX export

In [ ]:
import fw_infer, fw_export
model = fw_unet2d.load_model(CKPT)
print(fw_infer.validate(model, 'LTE_B71_617', seed=99))   # ~3 min: one CPU FDTD vs the surrogate
fw_export.export_onnx(model, fw_export.WEB / 'fw_unet2d.onnx'); print('exported', fw_export.WEB / 'fw_unet2d.onnx')

In [ ]:
from IPython.display import Image
Image('out/fw_validate/g2_LTE_B71_617.png')

### 8 · Save to Drive — flush + verify (so it can't vanish)

In [ ]:
import shutil, glob
from google.colab import drive
shutil.copytree(OUT, '/content/drive/MyDrive/fw_data_indoor', dirs_exist_ok=True)
shutil.copy(CKPT, '/content/drive/MyDrive/fw_unet2d_indoor.pt')
drive.flush_and_unmount(); drive.mount('/content/drive')          # forces the upload to complete
print(len(glob.glob('/content/drive/MyDrive/fw_data_indoor/shard_*.npz')), 'shards on Drive (verified)')